# Stage 1: COD10K Dataset Inspection & DataLoader Verification

This notebook provides an interactive walkthrough of:
1. **Dataset Structure Discovery** - Understanding COD10K folder layout
2. **Metadata Extraction** - Building a comprehensive dataset statistics table
3. **Visual Inspection** - Viewing raw and transformed samples
4. **DataLoader Verification** - Confirming batch format and tensor shapes

## Context: Camouflage-Aware Medical Image Segmentation

**Stage 1** trains two camouflage expert models on COD10K:
- **CU1**: Standard U-Net trained on camouflage objects (COD10K)
- **CUA1**: Attention U-Net trained on camouflage objects (COD10K)

These models learn to detect hard-to-see objects, which later guides medical image segmentation in Stage 2.

**This notebook focuses on**: Setting up the data pipeline (no model training yet)

## Section 1: Setup & Environment Configuration

In [ ]:
# Set up path to import our modules
import sys
from pathlib import Path

# Add src directory to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Src path: {src_path}")
print(f"Python path updated: {src_path in sys.path}")

In [ ]:
# Import standard libraries
import os
import warnings
warnings.filterwarnings('ignore')

# Import data processing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from tqdm import tqdm

# Import PyTorch
import torch
import torch.utils.data

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

print("\n✅ All libraries imported successfully!")

In [ ]:
# Import our custom modules
from config import COD10K_ROOT, OUTPUT_DIR, VISUALIZATION_DIR, CHECKPOINT_DIR, DEVICE
from config import DEFAULT_IMAGE_SIZE, DEFAULT_BATCH_SIZE, DEFAULT_VAL_RATIO

print("Project Configuration:")
print(f"  COD10K Root: {COD10K_ROOT}")
print(f"  Output Dir: {OUTPUT_DIR}")
print(f"  Visualization Dir: {VISUALIZATION_DIR}")
print(f"  Checkpoint Dir: {CHECKPOINT_DIR}")
print(f"  Device: {DEVICE}")
print(f"  Default Image Size: {DEFAULT_IMAGE_SIZE}")
print(f"  Default Batch Size: {DEFAULT_BATCH_SIZE}")

## Section 2: Dataset Folder Discovery & Validation

In [ ]:
# Check if COD10K dataset exists
print(f"Looking for COD10K dataset at: {COD10K_ROOT}")

if COD10K_ROOT.exists():
    print(f"\n✅ Dataset root found!")
    print(f"\nContents of {COD10K_ROOT}:")
    for item in sorted(COD10K_ROOT.iterdir()):
        if item.is_dir():
            print(f"  📁 {item.name}/")
            for subitem in sorted(item.iterdir()):
                if subitem.is_dir():
                    file_count = len(list(subitem.glob('*')))
                    print(f"     📁 {subitem.name}/ ({file_count} files)")
        else:
            print(f"  📄 {item.name}")
else:
    print(f"\n❌ Dataset not found at {COD10K_ROOT}")
    print("\nPlease download COD10K from:")
    print("  https://www.kaggle.com/datasets/aryehgod/camouflage-object-detection-cod10k")
    print(f"\nAnd extract it to: {COD10K_ROOT}")

## Section 3: Dataset Inspection Utility

In [ ]:
# Import inspection utilities
from utils import inspect_dataset

# Only run if dataset exists
if COD10K_ROOT.exists():
    print("Running comprehensive dataset inspection...\n")
    metadata_df, summary = inspect_dataset(
        root_dir=COD10K_ROOT,
        output_csv=VISUALIZATION_DIR.parent / 'cod10k_metadata.csv'
    )
else:
    print("Skipping inspection - dataset not found")
    metadata_df = None
    summary = None

## Section 4: Metadata Table & Summary Statistics

In [ ]:
if metadata_df is not None:
    print("Dataset Metadata Table:")
    print(f"Shape: {metadata_df.shape}")
    print(f"\nColumns: {list(metadata_df.columns)}")
    print(f"\nFirst 5 rows:")
    display(metadata_df.head())

In [ ]:
if metadata_df is not None:
    print("\n📊 Dataset Statistics by Split:")
    split_stats = metadata_df.groupby('split').agg({
        'filename': 'count',
        'image_width': ['min', 'mean', 'max'],
        'image_height': ['min', 'mean', 'max'],
        'mask_exists': 'sum',
    }).round(1)
    print(split_stats)
    
    print("\n🎯 Data Quality Summary:")
    print(f"Total samples: {len(metadata_df)}")
    print(f"Missing masks: {(~metadata_df['mask_exists']).sum()}")
    print(f"Corrupted images: {metadata_df['corrupted_image'].sum()}")
    print(f"Corrupted masks: {metadata_df['corrupted_mask'].sum()}")
    print(f"Shape mismatches: {metadata_df['shape_mismatch'].sum()}")
    print(f"Empty masks: {metadata_df['mask_empty'].sum()}")

## Section 5: Visualization Functions

In [ ]:
from utils import visualize_sample

# Helper function to display image-mask pairs
def show_samples_from_paths(image_paths, mask_paths, num_samples=3):
    """Display random image-mask pairs from file paths."""
    for i in range(min(num_samples, len(image_paths))):
        img = Image.open(image_paths[i]).convert('RGB')
        mask = Image.open(mask_paths[i]).convert('L')
        
        img_array = np.array(img, dtype=np.uint8)
        mask_array = np.array(mask, dtype=np.uint8)
        
        # Ensure binary mask
        mask_array = (mask_array > 127).astype(np.uint8) * 255
        
        visualize_sample(
            image=img_array,
            mask=mask_array[:, :, np.newaxis],
            title=f"Raw Sample {i+1}",
            show=True,
            save_path=VISUALIZATION_DIR / f'raw_sample_{i:02d}.png'
        )
        
print("Visualization functions loaded!")

In [ ]:
# Display raw samples if dataset exists
if metadata_df is not None and len(metadata_df) > 0:
    # Get train samples
    train_samples = metadata_df[metadata_df['split'] == 'train'].head(3)
    if len(train_samples) > 0:
        img_paths = [Path(p) for p in train_samples['image_path'].values]
        mask_paths = [Path(p) for p in train_samples['mask_path'].values]
        
        print("Displaying raw training samples...\n")
        show_samples_from_paths(img_paths, mask_paths, num_samples=3)

## Section 6: COD10KSegmentationDataset Implementation

In [ ]:
from datasets import COD10KSegmentationDataset

# Create training dataset
if COD10K_ROOT.exists():
    print("Creating COD10K training dataset...")
    train_dataset = COD10KSegmentationDataset(
        root_dir=COD10K_ROOT,
        split='train',
        image_size=DEFAULT_IMAGE_SIZE,
        normalize=False,
        augment=False,  # No augmentation for initial inspection
    )
    
    print(f"\n✅ Dataset loaded: {len(train_dataset)} samples")
    
    # Get a single sample
    sample = train_dataset[0]
    print(f"\nSample keys: {list(sample.keys())}")
    print(f"Image shape: {sample['image'].shape}")
    print(f"Mask shape: {sample['mask'].shape}")
    print(f"Image dtype: {sample['image'].dtype}")
    print(f"Mask dtype: {sample['mask'].dtype}")
    print(f"Filename: {sample['filename']}")
else:
    print("Cannot create dataset - COD10K not found")
    train_dataset = None

In [ ]:
# Visualize transformed samples
if train_dataset is not None:
    print("Displaying transformed tensor samples...\n")
    for i in range(3):
        sample = train_dataset[i]
        visualize_sample(
            image=sample['image'],
            mask=sample['mask'],
            title=f"Transformed Sample {i+1}: {sample['filename']}",
            show=True,
            save_path=VISUALIZATION_DIR / f'transformed_sample_{i:02d}.png'
        )

## Section 7: DataLoader Creation & Batch Verification

In [ ]:
from datasets import create_dataloaders

# Create DataLoaders
if COD10K_ROOT.exists():
    print("Creating DataLoaders...\n")
    train_loader, val_loader, test_loader = create_dataloaders(
        root_dir=COD10K_ROOT,
        batch_size=DEFAULT_BATCH_SIZE,
        image_size=DEFAULT_IMAGE_SIZE,
        num_workers=0,  # Use 0 for Jupyter, increase for production
        normalize=False,
        augment=False,  # For inspection, no augmentation
    )
    
    print(f"\n✅ DataLoaders created!")
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")
else:
    print("Cannot create dataloaders - dataset not found")
    train_loader = None
    val_loader = None
    test_loader = None

In [ ]:
# Get and verify a batch
if train_loader is not None:
    batch = next(iter(train_loader))
    
    print("\n📦 BATCH VERIFICATION")
    print("="*50)
    print(f"\nBatch keys: {list(batch.keys())}")
    print(f"\n📏 Tensor Shapes:")
    print(f"  Image: {batch['image'].shape}")
    print(f"  Expected: [16, 3, {DEFAULT_IMAGE_SIZE}, {DEFAULT_IMAGE_SIZE}]")
    print(f"  Match: {batch['image'].shape == torch.Size([DEFAULT_BATCH_SIZE, 3, DEFAULT_IMAGE_SIZE, DEFAULT_IMAGE_SIZE])}")
    
    print(f"\n  Mask: {batch['mask'].shape}")
    print(f"  Expected: [16, 1, {DEFAULT_IMAGE_SIZE}, {DEFAULT_IMAGE_SIZE}]")
    print(f"  Match: {batch['mask'].shape == torch.Size([DEFAULT_BATCH_SIZE, 1, DEFAULT_IMAGE_SIZE, DEFAULT_IMAGE_SIZE])}")
    
    print(f"\n📊 Data Types:")
    print(f"  Image dtype: {batch['image'].dtype}")
    print(f"  Mask dtype: {batch['mask'].dtype}")
    
    print(f"\n📈 Value Ranges:")
    print(f"  Image: [{batch['image'].min():.4f}, {batch['image'].max():.4f}]")
    print(f"  Mask: [{batch['mask'].min():.4f}, {batch['mask'].max():.4f}]")
    
    print(f"\n🎯 Mask Unique Values:")
    print(f"  {torch.unique(batch['mask'])}")
    print(f"  Expected: tensor([0., 1.])")
    
    print(f"\n✅ BATCH VERIFICATION COMPLETE")
    print("="*50)

In [ ]:
# Visualize a full batch
if train_loader is not None:
    print("\nVisualizing batch samples...\n")
    
    batch = next(iter(train_loader))
    
    # Plot first 4 samples in the batch
    fig, axes = plt.subplots(4, 3, figsize=(15, 12))
    
    for i in range(4):
        img = batch['image'][i].cpu().numpy().transpose(1, 2, 0)
        mask = batch['mask'][i].cpu().numpy().squeeze()
        
        # Normalize for display
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        
        # Image
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'Image {i}')
        axes[i, 0].axis('off')
        
        # Mask
        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].set_title(f'Mask {i}')
        axes[i, 1].axis('off')
        
        # Overlay
        overlay = img.copy()
        overlay[mask > 0.5] = overlay[mask > 0.5] * 0.5 + np.array([1, 0, 0]) * 0.5
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f'Overlay {i}')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(VISUALIZATION_DIR / 'batch_visualization.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f"✅ Batch visualization saved to {VISUALIZATION_DIR / 'batch_visualization.png'}")

## Section 8: Run Commands & Verification Summary

### Installation

```bash
# Navigate to project root
cd camouflage_medical_segmentation

# Create virtual environment (optional but recommended)
python -m venv venv
source venv/Scripts/activate  # Windows
source venv/bin/activate      # Linux/Mac

# Install dependencies
pip install -r requirements.txt
```

### Dataset Inspection

```bash
# Run comprehensive inspection
python inspect_dataset.py

# With custom path
python inspect_dataset.py --dataset-root /path/to/COD10K

# With more visualizations
python inspect_dataset.py --num-viz 10
```

### Jupyter Notebook

```bash
# Start notebook
jupyter notebook

# Open and run: notebooks/01_dataset_inspection.ipynb
```

In [ ]:
# Final verification summary
print("\n" + "="*70)
print("STAGE 1 DATASET PIPELINE VERIFICATION SUMMARY")
print("="*70)

print(f"\n✅ Configuration:")
print(f"   Dataset root: {COD10K_ROOT}")
print(f"   Exists: {COD10K_ROOT.exists()}")

if train_dataset is not None:
    print(f"\n✅ Dataset Loading:")
    print(f"   Train samples: {len(train_dataset)}")
    sample = train_dataset[0]
    print(f"   Image shape: {sample['image'].shape}")
    print(f"   Mask shape: {sample['mask'].shape}")

if train_loader is not None:
    print(f"\n✅ DataLoader Creation:")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")
    print(f"   Test batches: {len(test_loader)}")
    
    batch = next(iter(train_loader))
    print(f"\n✅ Batch Verification:")
    print(f"   Image: {batch['image'].shape} ✓" if batch['image'].shape[1:] == torch.Size([3, DEFAULT_IMAGE_SIZE, DEFAULT_IMAGE_SIZE]) else f"   Image: {batch['image'].shape} ✗")
    print(f"   Mask: {batch['mask'].shape} ✓" if batch['mask'].shape[1:] == torch.Size([1, DEFAULT_IMAGE_SIZE, DEFAULT_IMAGE_SIZE]) else f"   Mask: {batch['mask'].shape} ✗")
    print(f"   Values - Image: [{batch['image'].min():.2f}, {batch['image'].max():.2f}]")
    print(f"   Values - Mask: {torch.unique(batch['mask']).tolist()}")

print(f"\n" + "="*70)
print("🎉 DATASET PIPELINE READY FOR STAGE 1 TRAINING!")
print("="*70)
print("\nNext: Implement U-Net (CU1) and Attention U-Net (CUA1) models")